# Taller trader: resultados

Tablas y figuras generadas desde `results/`. Para regenerar los resultados, ver el README.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from rl_trader.prices import GENERATORS, make_series

## Generadores de precios

Una serie de cada generador (semilla fija).

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 6))
for ax, (name, _) in zip(axes.ravel(), GENERATORS.items()):
    ax.plot(make_series(name, 0, 500))
    ax.set_title(name)
    ax.grid()
plt.tight_layout()
plt.show()

## Resultados principales en test

100 series de test por generador; agentes con 5 semillas de entrenamiento.

In [ ]:
table = pd.read_csv(ROOT / "report" / "figures" / "tabla_resumen.csv")
table

In [ ]:
for name in ["fig_politica", "fig_aprendizaje", "fig_episodio", "fig_test"]:
    display(Image(filename=str(ROOT / "report" / "figures" / f"{name}.png")))

## Ajuste de hiperparámetros

Mejores parámetros por generador y algoritmo (validación): rejilla de α, γ y decaimiento de ε, y barrido de `epsilon_min` y número de episodios.

In [ ]:
import json

best = json.loads((ROOT / "results" / "tuning" / "sweep_best.json").read_text(encoding="utf-8"))
pd.json_normalize(best, sep="/").T

In [ ]:
pd.read_csv(ROOT / "results" / "tuning" / "sweep_summary.csv")

## Extensiones

Monte Carlo, entrenamiento mezclando generadores, número de tramos y comisión.

In [ ]:
for name in ["algoritmos", "sensibilidad"]:
    path = ROOT / "report" / "figures" / f"tabla_ext_{name}.csv"
    if path.is_file():
        display(Markdown(f"**{name}**"))
        display(pd.read_csv(path))

## Entrenamiento corto de ejemplo

Q-learning durante 300 episodios en el generador de reversión a la media, con los parámetros por defecto de `config/default.json`.

In [ ]:
from rl_trader.discretize import fit_discretizer
from rl_trader.experiments import load_config
from rl_trader.train import train

cfg = load_config(ROOT / "config" / "default.json")
disc = fit_discretizer("reversion_media", 50, cfg["n_steps"], cfg["n_bins"])
res = train("qlearning", "reversion_media", disc, 300, 0, cfg["n_steps"], **cfg["agent"])
plt.figure()
plt.plot(np.convolve(res.episode_rewards, np.ones(20) / 20, mode="valid"))
plt.xlabel("episodio")
plt.ylabel("recompensa (media móvil de 20)")
plt.grid()
plt.show()